In [ ]:
import sys, warnings, random, time
from pathlib import Path
import numpy as np, pandas as pd, networkx as nx
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.data_loader import get_dataset
from src.features.preprocessing import preprocess_node_features
from src.embeddings.graph_embeddings import compute_embeddings
from src.models.data import make_node_classification_data, make_link_prediction_data
from src.models import get_model, NodeClassificationTrainer, LinkPredictionTrainer
from src.evaluation.community import *
from src.sampling.graph_sampling import *
from src.visualization.distributions import plot_degree_distribution
from scipy.stats import entropy

In [ ]:
# Загрузка данных
rd = get_dataset('reddit', verbose=False)
G_full = rd['graph']
features_full = rd['features']
labels_full = rd['labels'].iloc[:, 0]

# Параметры сэмплирования
samplers = {
    'RandomNode': random_node_sampling,
    'ForestFire': forest_fire_sampling,
    'Snowball': snowball_sampling,
    'TIES': ties_sampling
}
target_sizes = [2000, 4000, 6000]
samp_graphs = {}      # (method, size) → граф

# Генерация семплов
print("Генерация семплов\n" + "-"*50)
for size in target_sizes:
    ratio = size / G_full.number_of_nodes()
    for sname, sampler in samplers.items():
        try:
            if sname == 'TIES':
                G_samp = sampler(G_full, target_num_nodes=size, seed=42)
            else:
                G_samp = sampler(G_full, ratio=ratio, seed=42)
            samp_graphs[(sname, size)] = G_samp
            print(f'{sname:12s} | target {size:5d} | nodes {G_samp.number_of_nodes():5d} | edges {G_samp.number_of_edges():7d}')
        except Exception as e:
            print(f'{sname} {size}: FAIL - {e}')

In [ ]:
deg_full = np.array([d for _, d in G_full.degree()])
kl_results = []

for sname in samplers:
    fig, axes = plt.subplots(1, len(target_sizes)+1, figsize=(5*(len(target_sizes)+1), 4))
    plot_degree_distribution(G_full, title='Full graph', ax=axes[0], log_scale=True)
    for i, size in enumerate(target_sizes):
        G_samp = samp_graphs[(sname, size)]
        plot_degree_distribution(G_samp, title=f'{sname} {size}', ax=axes[i+1], log_scale=True)
        deg_samp = np.array([d for _, d in G_samp.degree()])
        # KL-дивергенция
        hist_full, bin_edges = np.histogram(deg_full, bins=50, density=True)
        hist_samp, _ = np.histogram(deg_samp, bins=bin_edges, density=True)
        kl = entropy(hist_full + 1e-10, hist_samp + 1e-10)
        kl_results.append({'sampler': sname, 'size': size, 'kl_degree': kl})
        print(f'{sname} {size}: KL-divergence = {kl:.4f}')
    plt.tight_layout()
    plt.savefig(f'results/sampling_degree_{sname}.png')
    plt.show()

df_kl = pd.DataFrame(kl_results)
df_kl.to_csv('results/sampling_kl_divergence.csv', index=False)
df_kl

In [ ]:
nc_models = ['gatv2', 'graphsage']
lp_models = ['graphsage', 'gatv2']
all_results = []

for (sname, size), G_samp in tqdm(samp_graphs.items(), desc='NC & LP'):
    nodes_samp = sorted(G_samp.nodes())
    feat = features_full.reindex(nodes_samp).fillna(0).astype(np.float32)
    labs = labels_full.reindex(nodes_samp).dropna().astype(int)
    if len(labs) < 2:
        continue

    X = preprocess_node_features(feat, verbose=False)
    y = labs.values

    # NC
    data_nc = make_node_classification_data(G_samp, y, node_features=X, test_size=0.2, val_size=0.1, random_state=42)['data']
    for model_name in nc_models:
        try:
            mdl = get_model(model_name, num_features=data_nc.x.shape[1], num_classes=len(np.unique(y)), hidden_dim=64, seed=42)
            trainer = NodeClassificationTrainer(mdl, seed=42)
            trainer.fit(data_nc, epochs=100, early_stopping=20, verbose=False)
            met = trainer.evaluate(data_nc, mask=data_nc.test_mask)
            all_results.append({'task': 'NC', 'sampler': sname, 'size': size, 'model': model_name,
                                'accuracy': met['accuracy'], 'macro_f1': met['f1_macro']})
        except Exception as e:
            print(f'NC FAIL {sname} {size} {model_name}: {e}')

    # LP
    data_lp = make_link_prediction_data(G_samp, node_features=X, test_ratio=0.1, val_ratio=0.05, random_state=42)['data']
    for model_name in lp_models:
        try:
            mdl = get_model(model_name, num_features=data_lp.x.shape[1], num_classes=2, hidden_dim=64, seed=42)
            trainer = LinkPredictionTrainer(mdl, seed=42)
            trainer.fit(data_lp, epochs=80, early_stopping=10, verbose=False)
            met = trainer.evaluate(data_lp, stage='test')
            all_results.append({'task': 'LP', 'sampler': sname, 'size': size, 'model': model_name,
                                'auc': met['auc'], 'ap': met['ap']})
        except Exception as e:
            print(f'LP FAIL {sname} {size} {model_name}: {e}')

In [ ]:
for (sname, size), G_samp in tqdm(samp_graphs.items(), desc='CD'):
    nodes_samp = sorted(G_samp.nodes())
    labs = labels_full.reindex(nodes_samp).dropna().astype(int)
    if len(labs) < 2:
        continue
    true_labels = labs.to_dict()
    n_clusters = len(set(true_labels.values()))

    # Эмбеддинги для Node2Vec (быстро)
    emb_n2v = compute_embeddings(G_samp, method='node2vec', dimensions=32,
                                 name=f"reddit_cd_{sname}_{size}", seed=42,
                                 walk_length=10, num_walks=5, window=5)

    # Louvain
    part = run_louvain(G_samp)
    met = evaluate_communities(true_labels, part)
    all_results.append({'task': 'CD', 'sampler': sname, 'size': size, 'method': 'Louvain',
                        'nmi': met['nmi'], 'ari': met['ari'], 'n_communities': len(set(part.values()))})

    # Leiden
    part = run_leiden(G_samp, seed=42, resolution=0.01)
    met = evaluate_communities(true_labels, part)
    all_results.append({'task': 'CD', 'sampler': sname, 'size': size, 'method': 'Leiden',
                        'nmi': met['nmi'], 'ari': met['ari'], 'n_communities': len(set(part.values()))})

    # Node2Vec+KMeans
    part = run_kmeans_emb(emb_n2v, n_clusters=n_clusters, random_state=42)
    met = evaluate_communities(true_labels, part)
    all_results.append({'task': 'CD', 'sampler': sname, 'size': size, 'method': 'Node2Vec+KMeans',
                        'nmi': met['nmi'], 'ari': met['ari'], 'n_communities': n_clusters})

    # HDBSCAN
    part_hdb = run_hdbscan(G_samp, emb_n2v, min_cluster_size=5, metric='euclidean', seed=42)
    if part_hdb:
        met = evaluate_communities(true_labels, part_hdb)
        all_results.append({'task': 'CD', 'sampler': sname, 'size': size, 'method': 'HDBSCAN',
                            'nmi': met['nmi'], 'ari': met['ari'],
                            'n_communities': len(set(part_hdb.values()))})

In [ ]:
# Сохранение всех результатов
df_all = pd.DataFrame(all_results)
df_all.to_csv('results/sampling_all_tasks.csv', index=False)
df_all

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
tasks = [('NC', 'accuracy', 'Node Classification Accuracy'),
         ('LP', 'auc', 'Link Prediction AUC'),
         ('CD', 'nmi', 'Community Detection NMI')]

for ax, (task, metric, title) in zip(axes, tasks):
    subset = df_all[df_all['task'] == task]
    pivot = subset.pivot_table(values=metric, index='sampler', columns='size', aggfunc='mean')
    if pivot.empty:
        ax.text(0.5, 0.5, 'Нет данных', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        continue
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Target Nodes')
    ax.set_ylabel('Sampler')

plt.tight_layout()
plt.savefig('results/sampling_heatmaps.png')
plt.show()

In [ ]:
for task, metric, _ in tasks:
    print(f'\n--- {task} ({metric}) ---')
    sub = df_all[df_all['task'] == task]
    if sub.empty:
        print('нет данных')
        continue
    pivot = sub.pivot_table(values=metric, index='sampler', columns='size', aggfunc='mean')
    print(pivot.round(4))

# Graph Social Networks Analysis

Выпускная квалификационная работа на тему:  
**«Исследование графов больших социальных сетей методами машинного обучения»**

## Установка

```bash
git clone https://github.com/yungBaSe/graph-social-analysis.git
cd graph-social-analysis

python -m venv venv
source venv/bin/activate        # Windows: venv\Scripts\activate

pip install -r requirements.txt
```

## Структура проекта

- `src/data/` — загрузка и обработка датасетов.
- `src/embeddings/` — обучение эмбеддингов.
- `src/evaluation/` — реализация основных экспериментов.
- `src/features/` — вычисление структурных метрик и обработка признаков.
- `src/models/` — реализации классов обучаемых моделей.
- `src/sampling/` — методы сэмплирования.
- `src/visualization/` — визуализация графов.
- `notebooks/` — исследовательские ноутбуки.

## Данные и артефакты (игнорируются Git'ом)

- `data/raw/` — скачанные архивы
- `data/processed/` — кешированные графы, эмбеддинги, метрики
- `data/extracted/` — распакованные файлы из ZIP-архивов
- `figures/` — сохранённые визуализации

# Запуск

## Проверка базового функционала

```bash
jupyter notebook notebooks/demo/
```

## Интерактивный режим

```bash
jupyter notebook notebooks/
```

## Пакетный запуск всех ноутбуков 

```bash
export MPLBACKEND=Agg
jupyter nbconvert --execute --to notebook --inplace notebooks/*.ipynb
```

# Превью текущих итогов сравнения моделей в рамках каждой из задач

## Классификация вершин (Node Classification)

Accuracy / Macro‑F1 (среднее ± std по запускам).

| Датасет | Лучшая модель | Accuracy | Macro‑F1 |
|:---|:---|:---|:---|
| Cora | GATv2 | 0.854 ± 0.011 | 0.843 ± 0.006 |
| PubMed | GCNII | 0.886 ± 0.004 | 0.883 ± 0.004 |
| Twitch RU | GIN / GCNII | 0.753 ± 0.002 | 0.430 ± 0.001 |
| LastFM Asia | MLP | 0.745 ± 0.005 | 0.492 ± 0.008 |
| ogbn‑arxiv | GraphSAGE | 0.644 ± 0.000 | 0.408 ± 0.004 |

## Предсказание связей (Link Prediction)

AUC (среднее по запускам). Структурные эвристики приведены как базовый уровень.

| Датасет | Лучшая модель (GNN) | AUC модели | Лучшая эвристика | AUC эвристики |
|:---|:---|:---|:---|:---|
| Cora | Logistic | 0.820 | Adamic‑Adar | 0.697 |
| PubMed | GraphSAGE | 0.933 | Pref. Attach. | 0.720 |
| Facebook | GIN | 0.956 | Resource Alloc. | **0.988** |
| Twitch RU | GATv2 | 0.918 | Pref. Attach. | 0.897 |
| LastFM Asia | GAT | 0.805 | Adamic‑Adar | 0.837 |
| ogbn‑arxiv | GraphSAGE | 0.918 | Resource Alloc. | 0.895 |


## Обнаружение сообществ (Community Detection)

NMI / ARI на подграфах YouTube (100, 200, 400 ground‑truth сообществ).

| Размер | Лучший метод | NMI | ARI |
|:---|:---|:---|:---|
| 100 | HDBSCAN | 0.926 | 0.725 |
| 200 | HDBSCAN | 0.908 | 0.694 |
| 400 | HDBSCAN | 0.874 | 0.514 |

## Сэмплирование Reddit

Качество downstream‑задач на семплах (среднее по размерам 2000/4000/6000).

| Метод | NC Accuracy | LP AUC | CD NMI |
|:---|:---|:---|:---|
| Forest Fire | 0.628 | 0.534 | 0.322 |
| TIES | 0.121 | 0.628 | 0.557 |